In [ ]:
import os
import sys
sys.path.insert(0, os.path.expanduser("source"))

import json

import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from sentence_transformers.losses import ContrastiveLoss
from transformers import AutoTokenizer, TrainingArguments
from sklearn.utils.class_weight import compute_class_weight

from models import RoBERTaClassifierConfig, RoBERTaClassifier

from data_handling import (
    FlattenedMultiClassDataset, 
    FlattenedMultiLabelDataset,
    load_data,
    get_id_to_class,
    ChunkBatchSampler,
    custom_collate,
    get_custom_batched_tokenized
)

from evaluation import (
    compute_metrics, 
    compute_metrics_multilabel, 
    get_predictions, 
    get_multilabel_predictions_multiclass, 
    get_multiclass_predictions_multilabel, 
    evaluate_f1, 
    evaluate_ari, 
    evaluate_bcubed
)

from utils import to_coarse, multihot_to_list_of_classes, find_best_p
from training import HierarchicalContrastiveLoss, WeightedMultiClassTrainer, WeightedMultiLabelTrainer

In [ ]:
dataset_types = {"polynarrative": "multilabel", "CARDS": "multiclass"}
dataset_name = "CARDS"
dataset_type = dataset_types.get(dataset_name)

<h2>Load the dataset:</h2>

In [ ]:
exclude_other = False
other_vs_rest = False
dataset, (dataset_train, dataset_dev, dataset_test), id_to_class, class_to_id = load_data(
    dataset_name, 
    exclude_other, 
    other_vs_rest)

<p>If using a model that has been trained on a different dataset: </p>

In [ ]:
model_dataset_name = "polynarrative"
_, model_id_to_class, _ = load_dataset(model_dataset_name, False, False, return_torch_datasets=False)
model_class_to_id = {c: i for i, c in model_id_to_class.items()}

<h2>Training a classifier</h2>

<p>Turn each narrative id into a class: </p>

In [ ]:
if dataset_type == "multiclass":
    def map_narrative_to_class(ex):
        return {"labels": torch.tensor(id_to_class[ex["subnarrative"]])}

else:
    def map_narrative_to_class(ex):
        return {"labels": torch.zeros(len(id_to_class.keys())).index_fill_(0, torch.tensor([id_to_class[n] for n in set(ex["subnarrative"])]), 1)}

dataset = dataset.map(map_narrative_to_class)

<p>Compute weights for training: </p>

In [ ]:
if dataset_type == "multiclass":
    labels = np.array([id_to_class[sn] for sn in dataset["train"]["subnarrative"]])
    weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
    weights = torch.tensor(weights, dtype=torch.float)
else:
    train_labels = torch.vstack([l for l in dataset["train"]["labels"]])
    occurences = train_labels.sum(dim=0)
    non_occurences = train_labels.shape[0] - occurences
    weights = non_occurences / occurences
    weights = weights.sqrt() # smoothing, making sure no weights are too big due to extreme imbalance

In [ ]:
batch_size = 50 if dataset == "polynarrative" else 6
epochs = 10
aggregation = "attention"
max_length = 512
overlap_proportion = 0.4
stride_tokens = int(max_length * overlap_proportion)
model_name = "xlm-roberta-base"
n_layers_train = 2
lr = 1e-4

config = RoBERTaClassifierConfig(model_name=model_name, n_classes=len(id_to_class.keys()), aggregation=aggregation)
tokenizer = AutoTokenizer.from_pretrained(config.model_name, use_fast=True)
model = RoBERTaClassifier(config)

model_path = f"models/{dataset_name}/{model_name}_classifier_{n_layers_train if n_layers_train >= 0 else 'all'}_layers_lr_{lr}"

# Freeze some layers
if n_layers_train >= 0:
    # Freeze all parameters first
    for param in model.model.parameters():
        param.requires_grad = False
    
    if n_layers_train > 0:
        # Unfreeze last n_layers_train
        for layer in model.model.encoder.layer[-n_layers_train:]:
            for param in layer.parameters():
                param.requires_grad = True
else:
    # train everything
    for param in model.model.parameters():
        param.requires_grad = True

<p>Tokenize the dataset and split into batches by the amount of chunks: </p>

In [ ]:
def tokenize_with_sliding_window(ex):
    # return_overflowing_tokens splits long docs into multiple chunks
    enc = tokenizer(
        ex["text"],
        truncation=True,
        max_length=max_length,
        stride=stride_tokens,
        return_overflowing_tokens=True,
        padding="max_length",
    )
    # Will always be all zeroes
    enc.pop("overflow_to_sample_mapping")
    enc["labels"] = ex["labels"]
    return enc

train_tok = dataset["train"].map(tokenize_with_sliding_window, batched=False, remove_columns=dataset["train"].column_names)
dev_tok = dataset["dev"].map(tokenize_with_sliding_window, batched=False, remove_columns=dataset["dev"].column_names)

train_sampler = ChunkBatchSampler(train_tok, batch_size)
dev_sampler = ChunkBatchSampler(dev_tok, batch_size)

In [ ]:
args = TrainingArguments(
    output_dir=model_path,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    learning_rate=lr,
    weight_decay=0.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1_samples" if dataset_type=="multilabel" else "f1_macro",
    greater_is_better=True,
    fp16=True,
    report_to="none",
)

if dataset_type == "multiclass":
    trainer = WeightedMultiClassTrainer(
        class_weights=weights,
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        processing_class=tokenizer,
        compute_metrics=compute_metrics,
        train_sampler=train_sampler,
        dev_sampler=dev_sampler,
        data_collator=custom_collate
    )

else:
    trainer = WeightedMultiLabelTrainer(
        class_weights=weights,
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=dev_tok,
        processing_class=tokenizer,
        compute_metrics=compute_metrics_multilabel,
        train_sampler=train_sampler,
        dev_sampler=dev_sampler,
        data_collator=custom_collate
    )

trainer.train()
trainer.save_model()
print(f"Trained model and saved it to {model_path}")

<p>To load an already trained and saved model: </p>

In [ ]:
model_name = "xlm-roberta-base"
n_layers_train = 2
lr = 1e-4
model_path = f"models/{dataset_name}/{model_name}_classifier_{n_layers_train if n_layers_train >= 0 else 'all'}_layers_lr_{lr}"
model = RoBERTaClassifier.from_pretrained(model_path).to("cuda")
tokenizer = AutoTokenizer.from_pretrained(model_path)

<h2>Evaluation</h2>

In [ ]:
dataloader_dev = torch.utils.data.DataLoader(dev_tok, batch_sampler=dev_sampler, collate_fn=custom_collate)

<p>Run below for evaluation on the task the model itself is trained on: </p>

In [ ]:
coarse = False

def eval_f(min_p):
    truth, preds = get_predictions(model, dataloader_dev, dataset_type, extract_embeddings=False, min_p=min_p)
    f1, _ = evaluate_f1(truth, preds, class_to_id, dataset_type, coarse=coarse)
    return f1
        
if dataset_type == "multiclass":
    truth, preds = get_predictions(model, dataloader_dev, dataset_type, extract_embeddings=False)
    f1, cr = evaluate_f1(truth, preds, class_to_id, dataset_type, coarse=coarse)
    print(f1)
else:
    f1, t = find_best_p(eval_f, 0.01, 1.01, 0.01)
    print(f1, t)

<p>Run below for evaluating a multiclass classifier on a multi-label dataset: </p>

In [ ]:
dataloader_dev = torch.utils.data.DataLoader(dataset_dev, batch_size=1) # This evaluation needs this kind dataloader.

coarse=False

def eval_f(min_p):
    truth, preds = get_multiclass_predictions_multilabel(dataloader_dev, tokenizer, model, aggregation="union", min_p=min_p)
    if coarse:
        pred_ids = [{to_coarse(model_class_to_id[c]) for c in multihot_to_list_of_classes(p)} for p in preds]
    else:
        pred_ids = [multihot_to_list_of_classes(p) for p in preds]
    
    truth_ids = [{class_to_id[c] for c in multihot_to_list_of_classes(t)} for t in truth]

    precision, recall, f1 = evaluate_bcubed(truth_ids, pred_ids, coarse=coarse)
    return f1

f1, t = find_best_p(eval_f, 0, 1.01, 0.01)
print(model_path, coarse, f1, t)

<p>Run below for evaluating a multi-label classifier on a multiclass dataset: </p>

In [ ]:
coarse = False

truth, preds = get_multilabel_predictions_multiclass(dataloader_dev, model)
truth_ids = [class_to_id[t] for t in truth]

if coarse:
    preds = [to_coarse(model_class_to_id[p]) for p in preds]

ari = evaluate_ari(truth_ids, preds, class_to_id, coarse)

print(model_path, coarse, ari)